# 03 — Dựng năm bảng dimension từ Silver

### 1. Xem khóa và ngày nhận

Bắt đầu từ `delivery_silver`. Notebook gọi lần lượt hai hàm dimension của Prefect.

In [ ]:
from lakehouse_common import build_dimensions, courier_history
from scripts.check_project import load_notebook_steps

shared = load_notebook_steps()
DIM = shared.DIM
DIMS = shared.DIMS
SILVER = shared.SILVER

spark = shared.get_spark("Delivery - Dimensions")

In [ ]:
silver = spark.read.format("delta").load(SILVER)
silver.select("courier_id", "region_id", "aoi_id", "aoi_type", "accept_ts").show(5, truncate=False)

### 2. Dựng lịch sử shipper SCD2

Với mỗi shipper và ngày nhận, chọn vùng có nhiều đơn nhất. Khi bằng số
đơn, chọn `region_id` nhỏ hơn. Mỗi lần vùng đổi tạo một dòng lịch sử mới.
Fact dùng ngày nhận để ghép đúng khoảng `[effective_date, end_date)`.

In [ ]:
def build_dim_courier(spark):
    silver = spark.read.format("delta").load(SILVER)
    h = courier_history(silver)
    if h.filter("effective_date >= end_date").count():
        raise ValueError("SCD2: có dòng effective_date >= end_date")
    if h.groupBy("courier_id", "effective_date").count().filter("count > 1").count():
        raise ValueError("SCD2: trùng (courier_id, effective_date)")
    h.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(DIM)
    return {"dim_rows": h.count(), "couriers": h.select("courier_id").distinct().count()}

In [1]:
result = build_dim_courier(spark)
result

{'dim_rows': 226, 'couriers': 203}

In [ ]:
spark.read.format("delta").load(DIM).select(
    "courier_id", "region_id", "effective_date", "end_date", "is_active",
).show(5, truncate=False)

### 3. Dựng vùng, AOI, ngày và giờ

Vùng và AOI lấy khóa duy nhất từ Silver. Tọa độ hợp lệ dùng để tính tâm.
Dimension ngày phủ khoảng ngày nhận và giao. Dimension giờ có 24 dòng.

In [ ]:
def build_star_dimensions(spark):
    """Dựng dim_region, dim_aoi, dim_date, dim_hour (cùng dim_courier tạo thành star schema)."""
    silver = spark.read.format("delta").load(SILVER)
    out = {}
    for name, df in build_dimensions(silver).items():
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(DIMS[name])
        out[name] = df.count()
    return out

In [1]:
result = build_star_dimensions(spark)
result

{'dim_region': 17, 'dim_aoi': 8152, 'dim_date': 189, 'dim_hour': 24}